# 07 · Modelo da escolha do passe — Cover Shadow Value (CSV)

Fase 8b do CRISP-DM (D36, D37; critério C8). Em cada passe, o portador escolhe uma das linhas do conjunto de escolhas montado na fase 8a (`06_escolha_dados.ipynb`). Ajusta-se um logit condicional ("se passar, para quem?") e compara-se, com exatamente os mesmos passes e linhas, um modelo sem sombra (M0), um com a sombra virada para a bola (M1) um com a sombra pelo tronco (M2, a da v3) e um com as duas sombras ao mesmo tempo (M3).

**Critério C8:** a sombra pelo tronco ajuda a prever para quem vai o passe, fora da amostra, melhor do que sem sombra e do que a sombra virada para a bola.

Dados: SkillCorner Open Data (licença MIT), jogos 1925299 (Brisbane Roar – Perth Glory) e 1996435 (Sydney FC – Adelaide United). Código em `modelo_escolha.py`; números em `outputs/cover_shadow/modelo_escolha.json`.

## Definição dos passes e das linhas (D37)

- **Passe rasteiro:** `high_pass` diferente de verdadeiro e comprimento até 35 m. Os passes falhados (sem `high_pass`) entram se tiverem até 35 m.
- **Linhas:** as do conjunto de escolhas (sem fora de jogo nem cruzamentos) com comprimento até 35 m.
- **Momento da escolha:** frame alinhado até 0,4 s antes do passe.
- **Linhas sem pose:** a sombra de cada linha conta só os defesas que têm pose (valor mínimo) e a variável `linha_sem_pose` marca as linhas em que algum defesa que a podia afetar não tem pose.
- **Modelos:** M0 com `comprimento`, `avanco`, `valor_xt`, `recetor_marcado`, `dist_adversario` e `linha_sem_pose`; M1 = M0 + `sombra_bola`; M2 = M0 + `sombra_tronco`.

In [1]:
import sys

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from footy.paths import PROJECTS

sys.path.insert(0, str(PROJECTS / "cover_shadow"))  # para importar o código do projeto
import modelo_escolha as me

pd.options.display.float_format = "{:.3f}".format
r = me.calcular()
dados = r["dados"]
NOMES = {1925299: "Brisbane Roar – Perth Glory", 1996435: "Sydney FC – Adelaide United"}

## 1. Linhas sem pose: opção A e opção B, com o limite de 35 m

In [2]:
display(r["funil"].rename(columns={1925299: "1925299", 1996435: "1996435"}))
print("A: só passes cujas linhas têm todas pose. B: todos os passes com a linha escolhida no conjunto, com `linha_sem_pose`.")
print(f"Passes do modelo (B): {len(dados.drop_duplicates('passe'))}; linhas: {len(dados)}")
print("Linhas sem pose:", int(dados["linha_sem_pose"].sum()), f"({100 * dados['linha_sem_pose'].mean():.1f} %)")

jogo,1925299,1996435,total
passes com linhas no frame da escolha,805,759,1564
"rasteiros (não pelo ar, ≤ 35 m)",733,702,1435
"frame da escolha ≤ 0,4 s do passe",733,702,1435
B · linha escolhida no conjunto (≤ 35 m),728,686,1414
A · todas as linhas com pose,242,386,628


A: só passes cujas linhas têm todas pose. B: todos os passes com a linha escolhida no conjunto, com `linha_sem_pose`.
Passes do modelo (B): 1414; linhas: 10510
Linhas sem pose: 2005 (19.1 %)


## 2. Ajuste com os dois jogos juntos

Coeficientes do logit condicional (unidades das variáveis: metros para `comprimento`, `avanco` e `dist_adversario`; ganho de xT para `valor_xt`; 0/1 para `recetor_marcado` e `linha_sem_pose`; 0 a 1 para as sombras). `razao_de_chances` = exp(coef) por unidade; para uma sombra, uma unidade é uma linha totalmente tapada (sombra = 1) em comparação com uma totalmente livre (sombra = 0).

O M3 leva as duas sombras ao mesmo tempo. Mostra-se também como `sombra_bola` + `tronco_menos_bola` (sombra pelo tronco − sombra virada para a bola): é o mesmo modelo com outra parametrização (mesma verosimilhança); o coeficiente de `tronco_menos_bola` mede o que a sombra pelo tronco acrescenta à da bola.

In [3]:
for m, t in r["conjunto"].items():
    display(Markdown(f"**{m}** · log-verosimilhança {r['log_verosimilhanca'][m]:.1f}"))
    display(t)

**M0** · log-verosimilhança -2116.9

,coef,erro_padrao,ic95_inf,ic95_sup,razao_de_chances,rc_ic95_inf,rc_ic95_sup
comprimento,-0.160,0.006,-0.172,-0.148,0.852,0.842,0.862
avanco,0.068,0.004,0.060,0.077,1.071,1.062,1.080
valor_xt,3.915,2.327,-0.646,8.476,50.151,0.524,4797.138
recetor_marcado,-0.540,0.112,-0.760,-0.320,0.583,0.467,0.726
dist_adversario,0.187,0.009,0.169,0.205,1.205,1.184,1.227
linha_sem_pose,-0.255,0.135,-0.520,0.010,0.775,0.595,1.011


**M1** · log-verosimilhança -2047.8

,coef,erro_padrao,ic95_inf,ic95_sup,razao_de_chances,rc_ic95_inf,rc_ic95_sup
comprimento,-0.150,0.006,-0.162,-0.138,0.861,0.851,0.871
avanco,0.072,0.004,0.064,0.081,1.075,1.066,1.084
valor_xt,7.631,2.538,2.657,12.605,2060.709,14.254,297916.240
recetor_marcado,-0.558,0.113,-0.780,-0.336,0.572,0.458,0.715
dist_adversario,0.177,0.009,0.159,0.195,1.193,1.172,1.215
linha_sem_pose,-0.278,0.138,-0.547,-0.008,0.758,0.579,0.992
sombra_bola,-1.488,0.142,-1.767,-1.210,0.226,0.171,0.298


**M2** · log-verosimilhança -2039.0

,coef,erro_padrao,ic95_inf,ic95_sup,razao_de_chances,rc_ic95_inf,rc_ic95_sup
comprimento,-0.150,0.006,-0.162,-0.138,0.861,0.851,0.871
avanco,0.072,0.004,0.064,0.081,1.075,1.066,1.084
valor_xt,7.283,2.544,2.296,12.270,1455.638,9.937,213241.825
recetor_marcado,-0.547,0.113,-0.769,-0.325,0.579,0.463,0.723
dist_adversario,0.177,0.009,0.159,0.195,1.193,1.172,1.215
linha_sem_pose,-0.278,0.138,-0.548,-0.008,0.757,0.578,0.992
sombra_tronco,-1.750,0.162,-2.069,-1.432,0.174,0.126,0.239


**M3** · log-verosimilhança -2037.7

,coef,erro_padrao,ic95_inf,ic95_sup,razao_de_chances,rc_ic95_inf,rc_ic95_sup
comprimento,-0.149,0.006,-0.161,-0.137,0.861,0.851,0.872
avanco,0.072,0.004,0.064,0.081,1.075,1.066,1.084
valor_xt,7.565,2.559,2.549,12.580,1928.976,12.798,290742.793
recetor_marcado,-0.551,0.113,-0.774,-0.329,0.576,0.461,0.720
dist_adversario,0.176,0.009,0.158,0.194,1.192,1.171,1.214
linha_sem_pose,-0.278,0.138,-0.548,-0.009,0.757,0.578,0.991
sombra_bola,-0.414,0.262,-0.928,0.100,0.661,0.395,1.105
sombra_tronco,-1.362,0.295,-1.940,-0.783,0.256,0.144,0.457


**M3 (bola + diferença)** · log-verosimilhança -2037.7

,coef,erro_padrao,ic95_inf,ic95_sup,razao_de_chances,rc_ic95_inf,rc_ic95_sup
comprimento,-0.149,0.006,-0.161,-0.137,0.861,0.851,0.872
avanco,0.072,0.004,0.064,0.081,1.075,1.066,1.084
valor_xt,7.565,2.559,2.549,12.580,1928.974,12.798,290742.528
recetor_marcado,-0.551,0.113,-0.774,-0.329,0.576,0.461,0.720
dist_adversario,0.176,0.009,0.158,0.194,1.192,1.171,1.214
linha_sem_pose,-0.278,0.138,-0.548,-0.009,0.757,0.578,0.991
sombra_bola,-1.776,0.164,-2.097,-1.455,0.169,0.123,0.233
tronco_menos_bola,-1.362,0.295,-1.940,-0.783,0.256,0.144,0.457


In [4]:
desvios = dados[[*me.BASE, "sombra_bola", "sombra_tronco"]].std()
escala = r["conjunto"]["M2"][["coef"]].join(desvios.rename("desvio_padrao"))
escala["razao_de_chances_por_desvio_padrao"] = np.exp(escala["coef"] * escala["desvio_padrao"])
display(Markdown("**M2 · escala das variáveis** (desvio-padrão nas linhas do modelo e razão de chances por desvio-padrão)"))
display(escala)

**M2 · escala das variáveis** (desvio-padrão nas linhas do modelo e razão de chances por desvio-padrão)

,coef,desvio_padrao,razao_de_chances_por_desvio_padrao
comprimento,-0.150,8.173,0.294
avanco,0.072,14.755,2.896
valor_xt,7.283,0.016,1.124
recetor_marcado,-0.547,0.383,0.811
dist_adversario,0.177,6.002,2.888
linha_sem_pose,-0.278,0.393,0.897
sombra_tronco,-1.750,0.338,0.553


## 3. Fora da amostra

Ajuste num jogo e teste no outro, nos dois sentidos, com os mesmos passes e linhas nos quatro modelos. Por modelo: log-loss média por passe (menor = melhor), % de passes em que a linha escolhida é a mais provável (top-1) e está entre as 3 mais prováveis (top-3). Referências: escolha ao acaso entre as linhas do passe e um modelo só com o comprimento.

In [5]:
for chave, d in r["fora_da_amostra"].items():
    display(Markdown(f"**Ajuste em {d['treino']} ({NOMES[d['treino']]}) → teste em {d['teste']} ({NOMES[d['teste']]})**"))
    tabela = pd.DataFrame(d["modelos"]).T
    tabela.loc["ao acaso"] = {**d["ao_acaso"], "n_passes": tabela["n_passes"].iloc[0]}
    display(tabela.rename(index={"so_comprimento": "só comprimento"}).assign(
        top1=lambda t: 100 * t["top1"], top3=lambda t: 100 * t["top3"]).rename(columns={"top1": "top-1 (%)", "top3": "top-3 (%)"}))

**Ajuste em 1925299 (Brisbane Roar – Perth Glory) → teste em 1996435 (Sydney FC – Adelaide United)**

,log_loss,top-1 (%),top-3 (%),n_passes
M0,1.478,40.379,82.653,686.000
M1,1.430,43.440,84.111,686.000
M2,1.419,43.586,84.840,686.000
M3,1.420,43.440,84.694,686.000
só comprimento,1.693,31.050,73.178,686.000
ao acaso,1.987,14.101,42.302,686.000


**Ajuste em 1996435 (Sydney FC – Adelaide United) → teste em 1925299 (Brisbane Roar – Perth Glory)**

,log_loss,top-1 (%),top-3 (%),n_passes
M0,1.534,38.736,78.984,728.000
M1,1.483,40.934,80.082,728.000
M2,1.480,41.346,80.357,728.000
M3,1.479,41.621,80.495,728.000
só comprimento,1.744,29.808,71.703,728.000
ao acaso,1.971,14.428,43.214,728.000


M2 menos M1 e M0, e M3 menos M2, M1 e M0 (valores negativos = o primeiro modelo prevê melhor). Intervalo de confiança a 95 % por bootstrap sobre os passes do jogo de teste (5 000 repetições, semente fixa).

In [6]:
linhas = []
for chave, d in r["fora_da_amostra"].items():
    for nome in [k for k in d if "_menos_" in k]:
        linhas.append({"ajuste → teste": chave.replace("_para_", " → "), "comparação": nome.replace("_menos_", " − "), **d[nome]})
display(pd.DataFrame(linhas))

,ajuste → teste,comparação,diferenca,ic95_inf,ic95_sup
0,1925299 → 1996435,M2 − M1,-0.012,-0.024,0.001
1,1925299 → 1996435,M2 − M0,-0.060,-0.084,-0.035
2,1925299 → 1996435,M3 − M2,0.001,-0.004,0.007
3,1925299 → 1996435,M3 − M1,-0.010,-0.018,-0.003
4,1925299 → 1996435,M3 − M0,-0.058,-0.083,-0.034
5,1996435 → 1925299,M2 − M1,-0.003,-0.011,0.005
6,1996435 → 1925299,M2 − M0,-0.054,-0.074,-0.033
7,1996435 → 1925299,M3 − M2,-0.001,-0.002,-0.000
8,1996435 → 1925299,M3 − M1,-0.004,-0.011,0.004
9,1996435 → 1925299,M3 − M0,-0.055,-0.076,-0.034


## 4. Verificações

Matriz de correlação (Pearson) das variáveis nas linhas do modelo.

In [7]:
display(r["correlacoes"])
display(Markdown("**Sinais dos coeficientes (ajuste conjunto)**"))
sinais = pd.DataFrame({m: np.sign(t["coef"]).map({1.0: "+", -1.0: "−", 0.0: "0"}) for m, t in r["conjunto"].items()})
display(sinais)

,comprimento,avanco,valor_xt,recetor_marcado,dist_adversario,linha_sem_pose,sombra_bola,sombra_tronco
comprimento,1.000,0.063,0.065,-0.021,0.275,0.243,0.176,0.155
avanco,0.063,1.000,0.141,0.192,-0.457,0.162,0.174,0.139
valor_xt,0.065,0.141,1.000,0.203,-0.189,0.029,0.212,0.207
recetor_marcado,-0.021,0.192,0.203,1.000,-0.382,-0.024,0.057,0.052
dist_adversario,0.275,-0.457,-0.189,-0.382,1.000,-0.146,-0.224,-0.218
linha_sem_pose,0.243,0.162,0.029,-0.024,-0.146,1.000,0.067,0.056
sombra_bola,0.176,0.174,0.212,0.057,-0.224,0.067,1.000,0.888
sombra_tronco,0.155,0.139,0.207,0.052,-0.218,0.056,0.888,1.000


**Sinais dos coeficientes (ajuste conjunto)**

,M0,M1,M2,M3,M3 (bola + diferença)
avanco,+,+,+,+,+
comprimento,−,−,−,−,−
dist_adversario,+,+,+,+,+
linha_sem_pose,−,−,−,−,−
recetor_marcado,−,−,−,−,−
sombra_bola,NaN,−,NaN,−,−
sombra_tronco,NaN,NaN,−,−,NaN
tronco_menos_bola,NaN,NaN,NaN,NaN,−
valor_xt,+,+,+,+,+


## 5. Comparação com a SkillCorner

O `passing_option_score` da SkillCorner existe para as opções que ela deteta (fase 8a): uma opção só é registada se o score for superior a 0,6 durante pelo menos 0,3 s, ou se for o destino de um passe. Compara-se nos passes do modelo em que lista pelo menos duas opções e a linha escolhida está entre elas.

**O que é o score.** Não é uma probabilidade de escolha: varia entre 0 e 1 em cada opção, mas a soma das opções listadas num passe não é 1. Além disso, a escolha determina que opções estão listadas (o destino do passe entra sempre). Por isso o top-1 e o top-3 usam só a ordem do score; a log-loss calcula-se com o score dividido pela soma das opções listadas (`score normalizado`), o que dá uma distribuição mas não é uma probabilidade que a SkillCorner afirme. O M2 é o ajustado no outro jogo, restrito às mesmas opções, com as probabilidades renormalizadas.

In [8]:
total = len(dados.drop_duplicates("passe"))
print(f"Passes do modelo com pelo menos duas opções da SkillCorner: {r['passes_com_duas_opcoes_sc']} de {total}")
print(f"Passes na comparação (a linha escolhida também está listada): {r['skillcorner']['score_sc']['n_passes']}")
display(Markdown("**Score da SkillCorner nas opções listadas desses passes**"))
display(pd.Series(r["score_skillcorner"]).to_frame("valor"))

Passes do modelo com pelo menos duas opções da SkillCorner: 1177 de 1414
Passes na comparação (a linha escolhida também está listada): 1176


**Score da SkillCorner nas opções listadas desses passes**

,valor
minimo,0.032
maximo,0.999
soma_por_passe_mediana,2.396
soma_por_passe_min,0.949
soma_por_passe_max,6.306
score_medio_escolhida,0.864
score_medio_outras,0.802
passes,1176.000
linhas,3529.000


In [9]:
sk = r["skillcorner"]
tabela = pd.DataFrame({"M2 restrito": sk["M2_restrito"], "score SC (ordem)": sk["score_sc"],
                       "score SC normalizado": sk["score_sc_normalizado"], "ao acaso": sk["ao_acaso"]}).T
tabela["top1"], tabela["top3"] = 100 * tabela["top1"], 100 * tabela["top3"]
display(tabela.rename(columns={"top1": "top-1 (%)", "top3": "top-3 (%)"}))
display(Markdown("**Log-loss do M2 restrito menos a do score normalizado** (IC 95 % por bootstrap sobre os passes)"))
display(pd.Series(sk["M2_restrito_menos_score_normalizado"]).to_frame("valor").T)

,log_loss,top-1 (%),top-3 (%),n_passes
M2 restrito,1.103,47.959,94.048,1176.000
score SC (ordem),NaN,61.139,95.238,1176.000
score SC normalizado,1.032,61.139,95.238,1176.000
ao acaso,1.044,36.957,92.120,1176.000


**Log-loss do M2 restrito menos a do score normalizado** (IC 95 % por bootstrap sobre os passes)

,diferenca,ic95_inf,ic95_sup
valor,0.070,0.030,0.112
